# 01 — Identity and label invariance
Validates the axis-A identity $H(\bar p_g)=W(g)+D(g)$ (exactness, $D\ge 0$) and the label-free property of axis A.

In [1]:
# Notebook: 01_identity_and_label_invariance
# GUARD validation — shared setup
# Grayscale seaborn figures, dpi 600, saved as both PNG and PDF, no captions/titles.
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

RESULTS = os.path.join("..", "results")
FIG = os.path.join(RESULTS, "figures")
TAB = os.path.join(RESULTS, "tables")
os.makedirs(FIG, exist_ok=True)
os.makedirs(TAB, exist_ok=True)

# Grayscale aesthetic
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"] = "0.2"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["font.family"] = "DejaVu Sans"
GREYS = ["#111111", "#555555", "#888888", "#bbbbbb", "#dddddd"]

def savefig(fig, name):
    """Save a figure as PNG and PDF at dpi 600, no caption."""
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{name}.{ext}"), dpi=600, bbox_inches="tight")
    plt.close(fig)


In [2]:
# GUARD core metric (§2 of the research plan), inlined for a self-contained notebook.
EPS = 1e-12

def entropy(p, axis=-1):
    # Shannon entropy in natural log units.
    p = np.clip(p, EPS, 1.0)
    return -np.sum(p * np.log(p), axis=axis)

def js_divergence(q, p):
    # Jensen-Shannon divergence in base-2 (range [0, 1]).
    q = np.clip(q, EPS, 1.0); p = np.clip(p, EPS, 1.0)
    m = 0.5 * (q + p)
    return 0.5 * np.sum(q * np.log2(q / m)) + 0.5 * np.sum(p * np.log2(p / m))

def axis_A(P):
    # Homogeneity decomposition: H(pbar) = W + D  (BALD / Jensen-gap identity).
    pbar = P.mean(axis=0)
    W = entropy(P, axis=1).mean()      # within-member ambiguity (aleatoric)
    H_pbar = entropy(pbar)             # total marginal uncertainty
    D = H_pbar - W                     # between-member disagreement (epistemic) >= 0
    return W, D, H_pbar, pbar

def axis_B(P, a, K):
    # Contamination decomposition: C = JS(claim || belief), kappa = residual concentration.
    n = len(a)
    q = np.bincount(a, minlength=K).astype(float) / n   # claim (assigned-label) distribution
    pbar = P.mean(axis=0)                                # belief distribution
    C = js_divergence(q, pbar)
    r = np.clip(pbar - q, 0.0, None)                    # belief-exceeds-claim residual
    if r.sum() <= EPS:
        kappa = 0.0
    else:
        rt = r / r.sum()
        kappa = 1.0 - entropy(rt) / np.log(K)
    return C, kappa, q, pbar

def make_predictions(true_cats, K, signal=4.0, sigma=1.0, rng=None):
    # Predicted distribution peaked at each item's TRUE category:
    # the prediction reflects the product title, NOT the assigned label.
    if rng is None:
        rng = np.random.default_rng()
    n = len(true_cats)
    logits = rng.normal(0.0, sigma, size=(n, K))
    logits[np.arange(n), true_cats] += signal
    logits -= logits.max(axis=1, keepdims=True)
    P = np.exp(logits); P /= P.sum(axis=1, keepdims=True)
    return P

def individual_scores(P, a):
    # Per-item claim-belief mismatch used by aggregation baselines.
    n = len(a)
    return 1.0 - P[np.arange(n), a]


In [3]:
# Exp 1 — the identity H(pbar) = W + D holds exactly across random groups.
rng = np.random.default_rng(0)
K = 20
max_err, neg_D = 0.0, 0
errs = []
for _ in range(2000):
    n = int(rng.integers(3, 200))
    ncat = int(rng.integers(1, 8))
    base = rng.integers(0, K, size=ncat)
    true_cats = base[rng.integers(0, ncat, size=n)]
    P = make_predictions(true_cats, K, signal=float(rng.uniform(0.5, 6.0)),
                         sigma=float(rng.uniform(0.2, 2.0)), rng=rng)
    W, D, H_pbar, _ = axis_A(P)
    e = abs((W + D) - H_pbar); errs.append(e); max_err = max(max_err, e)
    if D < -1e-9:
        neg_D += 1
print(f"groups tested          : 2000")
print(f"max |(W+D) - H(pbar)|  : {max_err:.3e}")
print(f"groups with D < 0      : {neg_D}")


groups tested          : 2000
max |(W+D) - H(pbar)|  : 2.220e-16
groups with D < 0      : 0


In [4]:
# Exp 2 — axis A (W, D, H) is invariant to the assigned-label scheme; axis B responds.
rng = np.random.default_rng(7)
K = 20; n = 500
true_cats = rng.integers(0, K, size=n)
P = make_predictions(true_cats, K, signal=3.0, sigma=1.0, rng=rng)

schemes = {
    "clean":          true_cats.copy(),
    "random_diffuse": rng.integers(0, K, size=n),
    "structured":     np.full(n, (int(true_cats[0]) + 1) % K),
}
rows = []
for name, a in schemes.items():
    W, D, H, _ = axis_A(P)
    C, kappa, _, _ = axis_B(P, a, K)
    rows.append(dict(label_scheme=name, W=W, D=D, H_pbar=H, C=C, kappa=kappa))
df = pd.DataFrame(rows)
df.to_csv(os.path.join(TAB, "t01_label_invariance.csv"), index=False)
print(df.to_string(index=False))
print("\naxis-A drift across schemes: dW = %.2e, dD = %.2e"
      % (df.W.max() - df.W.min(), df.D.max() - df.D.min()))


  label_scheme       W       D  H_pbar        C    kappa
         clean 2.03105 0.96114 2.99219 0.002276 0.264287
random_diffuse 2.03105 0.96114 2.99219 0.010852 0.372311
    structured 2.03105 0.96114 2.99219 0.844265 0.018285

axis-A drift across schemes: dW = 0.00e+00, dD = 0.00e+00


In [5]:
# Figure — stacked W + D = H(pbar) for a range of group compositions (grayscale).
rng = np.random.default_rng(3)
K = 20
compositions = [1, 2, 3, 4, 6, 8]   # number of distinct true categories per group
Ws, Ds = [], []
for ncat in compositions:
    n = 400
    base = np.arange(ncat)
    true_cats = base[rng.integers(0, ncat, size=n)]
    P = make_predictions(true_cats, K, signal=3.0, sigma=1.0, rng=rng)
    W, D, H, _ = axis_A(P)
    Ws.append(W); Ds.append(D)

x = np.arange(len(compositions))
fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.bar(x, Ws, color=GREYS[2], edgecolor="black", linewidth=0.6, label="W (within)")
ax.bar(x, Ds, bottom=Ws, color=GREYS[0], edgecolor="black", linewidth=0.6, label="D (between)")
ax.set_xticks(x); ax.set_xticklabels(compositions)
ax.set_xlabel("Number of distinct true categories in group")
ax.set_ylabel("Group uncertainty (nats)")
ax.legend(frameon=False)
savefig(fig, "f01_identity_stacked")
print("saved f01_identity_stacked.{png,pdf}")


saved f01_identity_stacked.{png,pdf}
